# Week 11: Difference-in-Differences — A Natural Experiment

**QM 2023 — Statistics II / Data Analytics**

---

## Motivating Question

> *The 2008 financial crisis hit highly leveraged REITs harder than low-leverage firms. Can we measure the **causal impact** of the crisis on return volatility for high-debt REITs?*

We have a monthly firm-level panel of REIT returns and financial characteristics from 2006–2010. The 2008 Lehman Brothers collapse triggered a severe credit crunch. Highly leveraged REITs — those carrying the most debt relative to assets before the crisis — faced a disproportionate shock: refinancing became impossible, margin calls hit, and credit downgrades triggered forced selling. Low-leverage REITs were far more insulated.

This creates a natural experiment — and Difference-in-Differences (DiD) is the tool we use to exploit it.

---

## What You Will Learn

By the end of this demo, you will be able to:

1. **Explain** the DiD estimator in plain English and compute it by hand from a 2×2 table
2. **Identify** the parallel trends assumption and recognize when it holds or fails
3. **Estimate** a DiD regression model using OLS and interpret every coefficient
4. **Visualize** parallel trends in real data and assess assumption credibility
5. **Run a placebo test** to check whether your DiD result is spurious

---

## The Core DiD Logic (Plain English)

Difference-in-Differences answers the question: *"How much did the treated group change **relative to** the control group?"*

- **Difference 1** — Compare the treated group before and after the event → their change
- **Difference 2** — Compare the control group before and after the same event → their change
- **DiD** = (Treated change) − (Control change)

The key insight: whatever common forces affect everyone (inflation, market-wide trends, seasonality) should be the same for both groups. By subtracting the control group's change, we strip those out and isolate the treatment effect.

**Think of it as:** What happened to the treated group above and beyond what we would have expected based on how the control group moved?

In [ ]:
# =============================================================================
# ALL IMPORTS — Run this cell first
# =============================================================================
import pandas as pd                        # DataFrames — loading, filtering, grouping data
import numpy as np                         # Numerical operations — means, exp(), arrays
import matplotlib.pyplot as plt            # Plotting — line charts, bar charts, scatter plots
import matplotlib.patches as mpatches      # Custom legend patches for annotated plots
import seaborn as sns                      # Statistical plotting — heatmaps, styled charts
import statsmodels.formula.api as smf      # OLS regression using R-style formulas
from pathlib import Path                   # File paths that work on Windows and Mac

# Inline figures (display plots directly in the notebook)
%matplotlib inline

# Figure style — consistent look for all plots
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.size"] = 11

# =============================================================================
# COLOR PALETTE — Consistent colors used throughout the notebook
# =============================================================================
COLOR_TREATED = "steelblue"    # High-leverage firms (Treated = 1)
COLOR_CONTROL = "tomato"       # Low-leverage firms  (Treated = 0)
COLOR_CRISIS  = "red"          # Crisis date vertical line
COLOR_DID     = "goldenrod"    # DiD coefficient highlight
COLOR_PLACEBO = "mediumpurple" # Placebo test results

# Repo root and output folder for saved figures (cwd should be Week-11-Demo)
ROOT = Path(".").resolve()
OUTPUT_DIR = ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Imports complete. Ready to go!")
print()
print("Color scheme for this notebook:")
print("  Blue (steelblue)  = High-leverage / Treated group")
print("  Red  (tomato)     = Low-leverage / Control group")
print("  Gold (goldenrod)  = DiD causal effect highlight")

---

## Section 1: The DiD Idea — Toy Example

Before we touch the real data, let's build intuition with a tiny example you can check by hand.

**Setup:** A new housing regulation (the "treatment") hits two groups of REIT investors. We observe their monthly returns in one period before and one period after the regulation takes effect.

| Group | Before | After | Change |
|-------|--------|-------|--------|
| Treated | 8% | 3% | −5% |
| Control | 7% | 4% | −3% |

**Question:** Is the −5% decline entirely caused by the regulation? Or did the whole market fall?

- The whole market fell by −3% (that's what the control group shows).
- The treated group fell by −5%, which is **−2% more** than the market.
- DiD = −5% − (−3%) = **−2%** — that's the causal effect of the regulation on treated investors.

In [ ]:
# =============================================================================
# TOY EXAMPLE — 2×2 DiD by hand
# =============================================================================

# Define the four cells of the 2×2 table
treated_before = 0.08   # Treated group, before period
treated_after  = 0.03   # Treated group, after period
control_before = 0.07   # Control group, before period
control_after  = 0.04   # Control group, after period

# Step 1 — Compute each group's before-after change
change_treated = treated_after - treated_before   # How much did treated group change?
change_control = control_after - control_before   # How much did control group change?

# Step 2 — DiD = difference of differences
did_estimate = change_treated - change_control    # Treated change minus control change

# Print each step clearly
print("=== Toy Example: 2×2 Difference-in-Differences ===")
print(f"Treated group:  Before = {treated_before:.0%},  After = {treated_after:.0%}")
print(f"Control group:  Before = {control_before:.0%},  After = {control_after:.0%}")
print()
print(f"Change in Treated group = {treated_after:.0%} − {treated_before:.0%} = {change_treated:.0%}")
print(f"Change in Control group = {control_after:.0%} − {control_before:.0%} = {change_control:.0%}")
print()
print(f"DiD Estimate = {change_treated:.0%} − ({change_control:.0%}) = {did_estimate:.0%}")
print()
print("The regulation caused an EXTRA -2% return for the treated group.")
print("Without DiD we would have attributed ALL of the -5% to the treatment — that's wrong!")

# Build a neat 2×2 DataFrame to show the table
toy_table = pd.DataFrame(
    {
        "Before": [treated_before, control_before],
        "After":  [treated_after,  control_after],
        "Change": [change_treated, change_control],
    },
    index=["Treated", "Control"]
)
toy_table["Before"] = toy_table["Before"].map("{:.0%}".format)
toy_table["After"]  = toy_table["After"].map("{:.0%}".format)
toy_table["Change"] = toy_table["Change"].map("{:.0%}".format)

print()
print("=== 2×2 Table ===")
print(toy_table.to_string())
print(f"\nDiD = Change(Treated) − Change(Control) = -5% − (-3%) = {did_estimate:.0%}")

In [ ]:
# =============================================================================
# TOY EXAMPLE — Simple 2-line plot (Before → After)
# =============================================================================

fig, ax = plt.subplots(figsize=(7, 5))

# Two time points: Before (0) and After (1)
time_points = [0, 1]
time_labels = ["Before", "After"]

# Plot Treated line (blue, solid)
ax.plot(time_points, [treated_before, treated_after],
        color=COLOR_TREATED, linewidth=2.5, marker="o", markersize=9,
        label="Treated group")

# Plot Control line (red, dashed)
ax.plot(time_points, [control_before, control_after],
        color=COLOR_CONTROL, linewidth=2.5, marker="s", markersize=9,
        linestyle="--", label="Control group")

# Annotate the DiD gap with an arrow
ax.annotate("", xy=(1, treated_after), xytext=(1, control_after),
            arrowprops=dict(arrowstyle="<->", color="purple", lw=2))
ax.text(1.05, (treated_after + control_after) / 2,
        f"DiD = {did_estimate:.0%}",
        color="purple", fontsize=12, va="center", fontweight="bold")

# Labels and formatting
ax.set_xticks(time_points)
ax.set_xticklabels(time_labels, fontsize=12)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.set_ylabel("Monthly Return", fontsize=12)
ax.set_title("Toy Example: DiD Visualized", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("Key insight: the SLOPE of the treated line is steeper than the control line.")
print("That gap in slopes = DiD estimator.")

### Interpretation Checkpoint: The Toy Example

Before moving on, make sure you can answer these questions:

- **Why can't we just look at the treated group's −5% decline and call it the causal effect?**
  Because the whole market fell by −3%. The −5% is a mix of market-wide decline AND the treatment effect.

- **What does DiD "difference out"?**
  Common trends — anything that affected both groups equally (market conditions, seasonality, etc.).

- **If the control group had fallen by −5% too, what would the DiD estimate be?**
  Zero! That would mean the treatment had no extra effect beyond the common decline.

This logic is exactly the same when we move to real data — the numbers get more complex, but the intuition is identical.

---

## Section 2: The Parallel Trends Assumption

DiD rests on one critical assumption: **Parallel Trends**.

**What it means (plain English):**
> *If the treatment had never happened, the treated group and the control group would have moved up and down together — in parallel — over time.*

We can never prove this directly (we can't observe the counterfactual). But we can:

1. **Check the pre-period:** If both groups tracked each other before the event, that's strong evidence they would have continued to do so.
2. **Run a placebo test:** Apply the DiD to a fake event date — if we find a spurious "effect", our parallel trends assumption is suspect.

The plots below show the difference between data where parallel trends **holds** and where it **fails**.

In [ ]:
# =============================================================================
# PARALLEL TRENDS — Synthetic illustration
# =============================================================================

# Shared time axis: 6 periods (3 before, 3 after)
time = [1, 2, 3, 4, 5, 6]       # Periods 1-3 = before,  4-6 = after
event_period = 3.5               # Event happens between period 3 and 4

# --- Scenario A: Parallel Trends HOLDS ---
# Both groups trend downward at the same rate before the event;
# treated group then takes an extra hit after the event.
holds_treated = [0.10, 0.09, 0.08, 0.04, 0.03, 0.02]   # drops extra after event
holds_control = [0.07, 0.06, 0.05, 0.04, 0.03, 0.02]   # continues same slope

# --- Scenario B: Parallel Trends FAILS ---
# Before the event the treated group is already trending differently from control.
# This pre-existing difference will contaminate our DiD estimate.
fails_treated = [0.10, 0.07, 0.04, 0.00, -0.03, -0.06]  # steeper pre-trend
fails_control = [0.07, 0.06, 0.05,  0.04,  0.03,  0.02]  # flat-ish

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for ax, treated_vals, control_vals, title, subtitle in [
    (axes[0], holds_treated, holds_control,
     "Scenario A: Parallel Trends HOLDS",
     "Pre-period lines are parallel → DiD is credible"),
    (axes[1], fails_treated, fails_control,
     "Scenario B: Parallel Trends FAILS",
     "Pre-period trends diverge → DiD is biased"),
]:
    # Plot treated (blue solid) and control (red dashed)
    ax.plot(time, treated_vals, color=COLOR_TREATED, linewidth=2.5,
            marker="o", markersize=8, label="Treated")
    ax.plot(time, control_vals, color=COLOR_CONTROL, linewidth=2.5,
            marker="s", markersize=8, linestyle="--", label="Control")

    # Vertical dashed line at event
    ax.axvline(x=event_period, color="gray", linestyle=":", linewidth=2)
    ax.text(event_period + 0.05, max(treated_vals) * 0.95, "Event",
            color="gray", fontsize=10)

    # Shade the pre-period region
    ax.axvspan(1, event_period, alpha=0.06, color="green", label="Pre-period")

    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.text(0.5, -0.12, subtitle, transform=ax.transAxes,
            ha="center", fontsize=10, style="italic", color="gray")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
    ax.set_xlabel("Period", fontsize=11)
    ax.set_ylabel("Return", fontsize=11)
    ax.legend(fontsize=10)

plt.suptitle("The Parallel Trends Assumption", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("Scenario A: Lines are parallel in the pre-period. DiD cleanly isolates the event effect.")
print("Scenario B: Treated group was ALREADY trending differently. DiD would over-estimate the effect.")

### Interpretation Checkpoint: Parallel Trends

**What to look for in the pre-period (green shaded region):**

- Are both lines running roughly in parallel (similar slopes, similar ups and downs)?
- If yes → parallel trends assumption is plausible, DiD is credible.
- If no → DiD may be biased, and you need to find a better control group or use a different method.

**Why this matters so much:**
DiD assumes that if the treatment had never happened, the treated group would have followed the *same trajectory* as the control group. In Scenario B, the treated group was already declining faster — DiD would attribute that pre-existing decline to the treatment, which is wrong.

In Section 7, we'll make this same plot with the real REIT data and check whether our actual data looks more like Scenario A or B.

---

## Section 3: Load and Explore the Real REIT Data

Now we move to real data. We have a monthly firm-level REIT panel from January 2006 through mid-2010 — spanning the lead-up to and the recovery from the 2008 financial crisis.

**Columns:**

| Column | Description |
|--------|-------------|
| `permno` | CRSP firm identifier |
| `month` | Year-month (YYYY-MM, e.g. 2008-09) |
| `usdret` | Monthly stock return (USD) |
| `realizedvol` | Monthly realized return volatility |
| `log_vol` | Natural log of realized volatility (our outcome variable) |
| `debt_at` | Total debt / total assets (leverage ratio) |
| `lnmcap` | Log market capitalization (size control) |
| `beta` | Market beta |
| `Treated` | 1 if firm's pre-crisis debt_at > median leverage (high leverage) |
| `Post` | 1 if month ≥ September 2008 (crisis cutoff) |

The crisis peak is widely dated to **September 2008** — when Lehman Brothers collapsed.

**Treatment assignment:** Firms are classified as Treated = 1 if their average pre-crisis leverage (`debt_at`) exceeded the cross-sectional median. This is a firm-level, pre-determined characteristic — it does not change over time and cannot be affected by the crisis itself.

In [ ]:
# =============================================================================
# LOAD DATA — bundled CSV in this repository (data/)
# =============================================================================

data_path = ROOT / "data" / "reit_firm_panel.csv"

# Load the CSV
df = pd.read_csv(data_path)

# Parse month as datetime (YYYY-MM format → first day of each month)
df["month"] = pd.to_datetime(df["month"], format="%Y-%m")

# Quick shape and type check
print("=== Dataset Overview ===")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Date range: {df['month'].min().strftime('%Y-%m')} to {df['month'].max().strftime('%Y-%m')}")
print(f"Unique firms (permno): {df['permno'].nunique()}")
print()
print("=== Data Types ===")
print(df.dtypes)
print()
print("=== First 8 Rows ===")
print(df[["permno", "month", "usdret", "realizedvol", "log_vol", "debt_at", "Treated", "Post"]].head(8).to_string(index=False))

In [ ]:
# =============================================================================
# DATA VALIDATION — Check quality before analyzing
# =============================================================================

# 1. Missing values: Are there any gaps in our key variables?
print("=== Missing Values ===")
key_cols = ["permno", "month", "log_vol", "debt_at", "Treated", "Post"]
for col in key_cols:
    n_missing = df[col].isnull().sum()
    print(f"  {col:15s}: {n_missing} missing  {'✓' if n_missing == 0 else '⚠ CHECK THIS'}")

# 2. Treatment is time-invariant: Each firm should have ONE treatment status
print()
print("=== Treatment Consistency ===")
treatment_per_firm = df.groupby("permno")["Treated"].nunique()
all_consistent = (treatment_per_firm == 1).all()
print(f"  Each firm has exactly one Treated value: {all_consistent}  {'✓' if all_consistent else '⚠ PROBLEM'}")

# 3. Sample sizes per group
n_treated_firms  = df[df["Treated"] == 1]["permno"].nunique()
n_control_firms  = df[df["Treated"] == 0]["permno"].nunique()
n_pre  = df[df["Post"] == 0].shape[0]
n_post = df[df["Post"] == 1].shape[0]

print()
print("=== Sample Sizes ===")
print(f"  High-leverage firms (Treated=1): {n_treated_firms}")
print(f"  Low-leverage  firms (Treated=0): {n_control_firms}")
print(f"  Pre-crisis observations:  {n_pre}")
print(f"  Post-crisis observations: {n_post}")
print(f"  Total observations:       {df.shape[0]}")

# 4. Descriptive statistics for the outcome variable
print()
print("=== Outcome Variable: log_vol ===")
print(df["log_vol"].describe().to_string())

### Interpretation Checkpoint: Know Your Data

Before running any model, always verify:

- **No missing values** in key variables — otherwise your regression silently drops observations.
- **Treatment is time-invariant** — each firm's `Treated` status must be the same across all months. If it changed over time, we'd have a different research design (not standard DiD).
- **Balanced groups** — we want roughly similar numbers of treated and control firms. A 55/56 split is excellent.
- **Sufficient pre- and post-periods** — we need enough months before the crisis to check parallel trends and enough after to measure the effect.

Everything looks good — let's visualize the data.

In [ ]:
# =============================================================================
# PLOT — Average log_vol over time: High-leverage vs Low-leverage firms
# =============================================================================

# The 2008-09 crisis date
crisis_date = pd.Timestamp("2008-09-01")

# Compute average realized volatility per group per month
# (plot realizedvol for intuitive interpretation, then explain log transformation)
high_lev_monthly = (
    df[df["Treated"] == 1]
    .groupby("month")[["realizedvol", "log_vol"]]
    .mean()
    .reset_index()
)

low_lev_monthly = (
    df[df["Treated"] == 0]
    .groupby("month")[["realizedvol", "log_vol"]]
    .mean()
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left panel: realized volatility (raw, intuitive) ---
ax = axes[0]
ax.plot(high_lev_monthly["month"], high_lev_monthly["realizedvol"],
        color=COLOR_TREATED, linewidth=2, label="High leverage (Treated=1)")
ax.plot(low_lev_monthly["month"], low_lev_monthly["realizedvol"],
        color=COLOR_CONTROL, linewidth=2, linestyle="--", label="Low leverage (Treated=0)")
ax.axvline(x=crisis_date, color=COLOR_CRISIS, linestyle="--", linewidth=2, label="Sep 2008 (Lehman)")
ax.set_title("Realized Volatility over Time", fontsize=12, fontweight="bold")
ax.set_xlabel("Month", fontsize=11)
ax.set_ylabel("Avg Realized Volatility", fontsize=11)
ax.legend(fontsize=9)

# --- Right panel: log_vol (our regression outcome) ---
ax = axes[1]
ax.plot(high_lev_monthly["month"], high_lev_monthly["log_vol"],
        color=COLOR_TREATED, linewidth=2, label="High leverage (Treated=1)")
ax.plot(low_lev_monthly["month"], low_lev_monthly["log_vol"],
        color=COLOR_CONTROL, linewidth=2, linestyle="--", label="Low leverage (Treated=0)")
ax.axvline(x=crisis_date, color=COLOR_CRISIS, linestyle="--", linewidth=2, label="Sep 2008 (Lehman)")
ax.set_title("Log Realized Volatility over Time (Regression Outcome)", fontsize=12, fontweight="bold")
ax.set_xlabel("Month", fontsize=11)
ax.set_ylabel("Avg log(Realized Volatility)", fontsize=11)
ax.legend(fontsize=9)

# Save figure
fig01_path = OUTPUT_DIR / "fig01_volatility_over_time.png"
plt.savefig(fig01_path, dpi=150, bbox_inches="tight")
print(f"Figure saved: {fig01_path}")

plt.tight_layout()
plt.show()

### Interpretation Checkpoint: What Do You See?

Look at the two panels above and answer these questions:

- **Do both groups spike at the same time?** Yes — both high- and low-leverage firms show a massive volatility spike in late 2008. This is the common crisis shock (the `Post` coefficient will capture this).

- **Does one group spike MORE than the other?** Yes — the blue line (high leverage) rises higher than the red line (low leverage). That *extra* spike is what DiD will measure.

- **Why use log(vol) instead of raw volatility?** The left panel shows raw volatility is right-skewed — the spike is enormous compared to normal times. Taking the log (right panel) compresses the scale, makes residuals more symmetric, and lets us interpret coefficients as approximate percentage changes.

- **Do the pre-crisis lines track together?** In the green-shaded pre-period, both lines move roughly in parallel. This is early visual evidence for the parallel trends assumption — we'll test it more formally later.

---

## Section 4: Create DiD Indicators

To run a DiD regression, we need three indicator variables:

| Variable | Definition |
|----------|------------|
| `Treated` | 1 if the firm's pre-crisis leverage (`debt_at`) exceeded the median (high leverage), 0 otherwise |
| `Post` | 1 if the month is **on or after** the crisis date (Sep 2008), 0 otherwise |
| `DiD` | `Treated × Post` — the interaction term; this is the key variable |

**How is Treated assigned?**

Treatment is based on each firm's **pre-crisis leverage ratio** (`debt_at` = total debt / total assets). Firms above the median are classified as high-leverage (Treated = 1); firms below are low-leverage (Treated = 0). This pre-determined classification cannot be affected by the crisis.

- **Treated = 1 (high leverage):** ~55 firms — faced refinancing risk, margin calls, credit downgrades during the crisis
- **Treated = 0 (control, low leverage):** ~56 firms — carried less debt, more insulated from credit market disruption

Both `Treated` and `Post` are already in the CSV (pre-assigned). We just need to create the `DiD` interaction.

In [ ]:
# =============================================================================
# CREATE DiD INDICATOR
# =============================================================================

# Treated and Post are already in the CSV — pre-assigned based on pre-crisis leverage
# Treated = 1 if firm's pre-crisis debt_at > median (high leverage)
# Post    = 1 if month >= September 2008 (Lehman Brothers collapse)

# Define the crisis date for reference
crisis_date = pd.Timestamp("2008-09-01")

# DiD interaction term: Treated × Post
# This equals 1 ONLY for high-leverage firms in the post-crisis period
df["DiD"] = df["Treated"] * df["Post"]

# Count unique firms per group
n_treated_firms  = df[df["Treated"] == 1]["permno"].nunique()
n_control_firms  = df[df["Treated"] == 0]["permno"].nunique()

print("=== Treatment Assignment ===")
print(f"High-leverage firms (Treated=1): {n_treated_firms}")
print(f"Low-leverage  firms (Treated=0): {n_control_firms}")
print(f"Total firms:                     {df['permno'].nunique()}")
print()
print(f"Crisis cutoff: {crisis_date.strftime('%Y-%m')} (Lehman Brothers collapse)")
print()

# Verify the counts using a crosstab
print("=== Crosstab: Treated × Post (number of observations) ===")
crosstab = pd.crosstab(
    df["Treated"].map({0: "Low leverage (0)", 1: "High leverage (1)"}),
    df["Post"].map({0: "Before (0)", 1: "After (1)"}),
    margins=True,
)
print(crosstab)
print()
print("=== First rows with new DiD column ===")
print(df[["permno", "month", "log_vol", "debt_at", "Treated", "Post", "DiD"]].head(12).to_string(index=False))

---

## Section 5: Manual DiD Calculation

Before running a regression, always compute the DiD by hand. This keeps you honest — you can verify the regression coefficient matches your hand calculation.

The four group means give us the entire DiD estimate:

$$\hat{\delta}_{DiD} = (\bar{Y}_{Treated,After} - \bar{Y}_{Treated,Before}) - (\bar{Y}_{Control,After} - \bar{Y}_{Control,Before})$$

We use **log_vol** as the outcome — the log of monthly realized volatility.

In [ ]:
# =============================================================================
# MANUAL DiD — Compute the 4 group means explicitly (using log_vol)
# =============================================================================

# Separate the data into the 4 cells of the DiD table
treated_before_data = df[(df["Treated"] == 1) & (df["Post"] == 0)]["log_vol"]
treated_after_data  = df[(df["Treated"] == 1) & (df["Post"] == 1)]["log_vol"]
control_before_data = df[(df["Treated"] == 0) & (df["Post"] == 0)]["log_vol"]
control_after_data  = df[(df["Treated"] == 0) & (df["Post"] == 1)]["log_vol"]

# Compute the mean for each cell
mean_treated_before = treated_before_data.mean()   # High-leverage firms avg log_vol BEFORE crisis
mean_treated_after  = treated_after_data.mean()    # High-leverage firms avg log_vol AFTER crisis
mean_control_before = control_before_data.mean()   # Low-leverage firms avg log_vol BEFORE crisis
mean_control_after  = control_after_data.mean()    # Low-leverage firms avg log_vol AFTER crisis

# Compute the two differences
change_treated_firms = mean_treated_after - mean_treated_before   # High-leverage group change
change_control_firms = mean_control_after - mean_control_before   # Low-leverage group change

# Final DiD estimate
did_manual = change_treated_firms - change_control_firms

# Print every step with clear labels
print("=== Manual DiD Calculation (Step by Step) ===")
print("Outcome: log_vol (log of monthly realized volatility)")
print()
print(f"STEP 1 — Group means:")
print(f"  High-leverage (Treated), Before crisis: {mean_treated_before:+.4f}")
print(f"  High-leverage (Treated), After  crisis: {mean_treated_after:+.4f}")
print(f"  Low-leverage  (Control), Before crisis: {mean_control_before:+.4f}")
print(f"  Low-leverage  (Control), After  crisis: {mean_control_after:+.4f}")
print()
print(f"STEP 2 — Before-after change for each group:")
print(f"  High-leverage change = {mean_treated_after:+.4f} − {mean_treated_before:+.4f} = {change_treated_firms:+.4f}")
print(f"  Low-leverage  change = {mean_control_after:+.4f} − {mean_control_before:+.4f} = {change_control_firms:+.4f}")
print()
print(f"STEP 3 — Difference-in-Differences:")
print(f"  DiD = High-leverage change − Low-leverage change")
print(f"  DiD = {change_treated_firms:+.4f} − ({change_control_firms:+.4f}) = {did_manual:+.4f}")
print()
print(f"  → exp({did_manual:.4f}) ≈ {np.exp(did_manual):.3f}")
print(f"  → High-leverage firms experienced approximately {(np.exp(did_manual)-1)*100:.1f}% MORE")
print(f"     volatility amplification than low-leverage peers (above common crisis shock).")

In [ ]:
# =============================================================================
# 2×2 HEATMAP — Visualize the DiD table as a color-coded grid
# =============================================================================

# Build the 2×2 matrix of group means
heatmap_data = pd.DataFrame(
    {
        "Before Crisis": [mean_control_before, mean_treated_before],
        "After Crisis":  [mean_control_after,  mean_treated_after],
    },
    index=["Low Leverage (Control)", "High Leverage (Treated)"]
)

fig, ax = plt.subplots(figsize=(8, 4))

# Draw the heatmap
sns.heatmap(
    heatmap_data,
    annot=True,            # Show numbers in each cell
    fmt=".3f",             # 3 decimal places
    cmap="YlOrRd",         # Yellow-Orange-Red: higher volatility = darker
    linewidths=2,
    linecolor="white",
    cbar_kws={"label": "Mean log(Realized Volatility)"},
    ax=ax,
    annot_kws={"fontsize": 14, "fontweight": "bold"},
)

ax.set_title("2×2 DiD Table: Mean log_vol by Group and Period",
             fontsize=13, fontweight="bold")
ax.set_ylabel("")
ax.set_xticklabels(ax.get_xticklabels(), fontsize=12)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=12, rotation=0)

# Annotate the DiD value
ax.text(1.0, -0.15, f"DiD = {did_manual:+.4f}  (extra volatility for high-leverage firms)",
        transform=ax.transAxes, ha="right", fontsize=11,
        color=COLOR_DID, fontweight="bold")

plt.tight_layout()

# Save figure
fig_heatmap_path = OUTPUT_DIR / "fig_did_heatmap.png"
plt.savefig(fig_heatmap_path, dpi=150, bbox_inches="tight")
print(f"Figure saved: {fig_heatmap_path}")

plt.show()

print()
print("Reading the heatmap:")
print("  - Darker cells = higher average log volatility")
print("  - Both groups get darker after the crisis (common shock)")
print(f"  - But the treated group's jump is LARGER by {did_manual:+.4f} log-points")
print(f"  - That extra darkening = the DiD causal estimate")

### Interpretation Checkpoint: The Manual DiD

**Check your understanding:**

- The **common crisis shock** (what happened to everyone) is the control group's change: approximately +1.31 log-points. That's enormous — the crisis hit all REITs hard.

- The **extra effect on high-leverage firms** is the DiD: approximately +0.13 log-points. This is smaller than the common shock, but it's statistically and economically meaningful.

- **Why does this matter economically?** exp(0.13) ≈ 1.14, meaning high-leverage REITs experienced about 14% more volatility amplification. The mechanism: refinancing risk, margin calls, and credit downgrades all hit leveraged firms harder.

- **The manual DiD should exactly match the regression coefficient β₃ in the next section.** If it doesn't, something is wrong with the model specification.

Now let's confirm this with a formal regression.

---

## Section 6: DiD Regression

The regression formulation of DiD is:

$$\text{log\_vol}_{it} = \alpha + \beta_1 \cdot \text{Treated}_i + \beta_2 \cdot \text{Post}_t + \beta_3 \cdot (\text{Treated}_i \times \text{Post}_t) + \varepsilon_{it}$$

This is just OLS with three right-hand-side variables — nothing exotic. The key is understanding what each coefficient means.

**A note on robust standard errors:**
We use `HC3` (heteroskedasticity-consistent) standard errors. Why? In real-world data, the variance of errors is rarely constant across all observations. Firms with higher volatility may have noisier measurements. HC3 standard errors adjust for this uneven variance, giving us more reliable confidence intervals and p-values. Think of it as a safety net — if heteroskedasticity is absent, HC3 gives nearly the same results as regular OLS; if it's present, HC3 protects us from overstating significance.

In [ ]:
# =============================================================================
# DiD REGRESSION — OLS with HC3 heteroskedasticity-robust standard errors
# =============================================================================

# Estimate: log_vol ~ Treated + Post + DiD
# smf (statsmodels formula API) lets us write the model as a formula string
# HC3 = heteroskedasticity-consistent standard errors (more reliable than default)
did_model = smf.ols("log_vol ~ Treated + Post + DiD", data=df).fit(cov_type="HC3")

# Print the full regression summary
print(did_model.summary())

# Extract key coefficient values for later use
intercept_coef    = did_model.params["Intercept"]  # α: control group baseline log_vol
treated_coef      = did_model.params["Treated"]    # β₁: permanent leverage gap
post_coef         = did_model.params["Post"]       # β₂: common crisis shock
did_coef          = did_model.params["DiD"]        # β₃: THE causal effect
did_se            = did_model.bse["DiD"]           # Standard error on DiD
did_pval          = did_model.pvalues["DiD"]       # p-value on DiD

print()
print("=== Key Coefficients ===")
print(f"Intercept (α):      {intercept_coef:+.4f}  (low-leverage baseline log_vol pre-crisis)")
print(f"Treated   (β₁):     {treated_coef:+.4f}  (pre-existing leverage gap)")
print(f"Post      (β₂):     {post_coef:+.4f}  (common crisis volatility spike — all firms)")
print(f"DiD       (β₃):     {did_coef:+.4f}   (SE = {did_se:.4f}, p = {did_pval:.4f})")
print()
print(f"Manual DiD (Section 5): {did_manual:+.4f}  ← Should match β₃ above!")
print()
print(f"R-squared: {did_model.rsquared:.4f}   N = {int(did_model.nobs)}")

### Interpreting Each Coefficient

The regression output has four numbers. Here is exactly what each one means:

**α (Intercept) ≈ −4.16** — *Low-leverage firms' average log_vol in the pre-crisis period*
This is the baseline: what low-leverage REITs' log volatility looked like before September 2008. Everything else is measured relative to this.

**β₁ (Treated) ≈ −0.01** — *Pre-existing leverage gap (not the causal effect)*
Before the crisis, did high-leverage firms have systematically different volatility from low-leverage firms? The near-zero coefficient suggests the two groups had similar volatility levels pre-crisis — which supports the parallel trends assumption.

**β₂ (Post) ≈ +1.3074 *** — *Common crisis shock shared by ALL firms*
The crisis caused ALL REIT volatility to spike by approximately 1.31 log-points — a massive common shock. This is Lehman Brothers hitting the entire market. It has nothing to do with leverage specifically; every REIT felt it.

**β₃ (DiD) ≈ +0.1329, p = 0.0007 *** ← THIS IS THE CAUSAL EFFECT**
After accounting for (1) the pre-existing leverage gap and (2) the common crisis shock, high-leverage REITs experienced an **additional** 0.133 log-point volatility increase.

> exp(0.1329) ≈ 1.14 → **high-leverage REITs had ~14% MORE volatility amplification** than low-leverage peers.

**The economic mechanism — the leverage amplification channel:**

- Refinancing risk: high-debt firms couldn't roll over short-term debt as credit markets froze
- Margin calls: leveraged positions were forcibly unwound, creating volatile price swings
- Credit downgrades: rating agencies downgraded high-leverage REITs, triggering institutional selling

In [ ]:
# =============================================================================
# COEFFICIENT PLOT — Bar chart with confidence intervals
# =============================================================================

# Extract all coefficients and confidence intervals
coef_names   = ["Intercept (α)", "Treated (β₁)", "Post (β₂)", "DiD (β₃ — causal effect)"]
coef_values  = did_model.params.values                          # Point estimates
coef_ci_low  = did_model.conf_int()[0].values                  # Lower 95% CI
coef_ci_high = did_model.conf_int()[1].values                  # Upper 95% CI

# Error bar sizes (half the CI width, for asymmetric bars)
err_low  = coef_values - coef_ci_low   # distance from point to lower CI
err_high = coef_ci_high - coef_values  # distance from point to upper CI

# Colors — highlight the DiD coefficient in gold
bar_colors = [COLOR_TREATED, COLOR_TREATED, COLOR_TREATED, COLOR_DID]

fig, ax = plt.subplots(figsize=(10, 5))

# Horizontal bar chart
y_positions = range(len(coef_names))
ax.barh(
    y_positions,
    coef_values,
    xerr=[err_low, err_high],
    color=bar_colors,
    edgecolor="black",
    linewidth=0.8,
    capsize=5,
    height=0.5,
)

# Zero reference line
ax.axvline(x=0, color="black", linewidth=1.2, linestyle="-")

# Label each bar with its value
for i, val in enumerate(coef_values):
    ax.text(val + 0.02, i, f"{val:+.4f}", va="center", fontsize=10)

ax.set_yticks(list(y_positions))
ax.set_yticklabels(coef_names, fontsize=11)
ax.set_xlabel("Coefficient Value (log volatility units)", fontsize=11)
ax.set_title("DiD Regression Coefficients with 95% Confidence Intervals\nOutcome: log(Realized Volatility)",
             fontsize=13, fontweight="bold")

# Annotation on the DiD bar
ax.text(0.97, 0.05, "← Leverage amplification effect\n   of the 2008 crisis",
        transform=ax.transAxes, ha="right", fontsize=9,
        color=COLOR_DID, fontstyle="italic")

plt.tight_layout()

# Save figure
fig02_path = OUTPUT_DIR / "fig02_did_coefficients.png"
plt.savefig(fig02_path, dpi=150, bbox_inches="tight")
print(f"Figure saved: {fig02_path}")

plt.show()

### Interpretation Checkpoint: Reading the Regression

**Verify your understanding:**

- **Does β₃ (DiD) match the manual calculation from Section 5?** It should be identical (or nearly so). This confirms the regression is doing exactly what we computed by hand — no magic, just arithmetic organized into a regression framework.

- **Which coefficient is the "causal effect"?** Only β₃ (DiD). The other coefficients are important for the model but are NOT causal:
  - β₁ (Treated) just measures pre-existing differences
  - β₂ (Post) measures the common shock to everyone

- **How do you read the confidence interval?** The gold bar shows the point estimate; the error bars show the 95% CI. If the CI doesn't cross zero, the effect is statistically significant at the 5% level.

- **Is the R² "good"?** R² tells us how much variation in log_vol the model explains. With only three variables (Treated, Post, DiD), a moderate R² is expected. We're not trying to explain ALL volatility — just isolate the leverage channel.

In [ ]:
# =============================================================================
# RESIDUAL DIAGNOSTICS — Check regression assumptions
# =============================================================================

# Extract residuals from the DiD model
residuals = did_model.resid                    # Actual log_vol − Predicted log_vol
fitted_values = did_model.fittedvalues         # Model's predicted log_vol

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# --- Panel 1: Histogram of residuals ---
ax = axes[0]
ax.hist(residuals, bins=40, color=COLOR_TREATED, edgecolor="white", alpha=0.7)
ax.axvline(x=0, color="black", linewidth=1.5, linestyle="--")
ax.set_xlabel("Residual (log_vol units)", fontsize=11)
ax.set_ylabel("Frequency", fontsize=11)
ax.set_title("Distribution of Residuals", fontsize=12, fontweight="bold")

# --- Panel 2: Residuals vs Fitted ---
ax = axes[1]
ax.scatter(fitted_values, residuals, alpha=0.3, s=15, color=COLOR_TREATED)
ax.axhline(y=0, color="black", linewidth=1.5, linestyle="--")
ax.set_xlabel("Fitted Values (predicted log_vol)", fontsize=11)
ax.set_ylabel("Residual", fontsize=11)
ax.set_title("Residuals vs. Fitted Values", fontsize=12, fontweight="bold")

# --- Panel 3: Residuals over time ---
ax = axes[2]
resid_by_month = df.copy()
resid_by_month["residual"] = residuals
monthly_resid = resid_by_month.groupby("month")["residual"].mean()
ax.plot(monthly_resid.index, monthly_resid.values, color=COLOR_TREATED, linewidth=1.5)
ax.axhline(y=0, color="black", linewidth=1.5, linestyle="--")
ax.axvline(x=pd.Timestamp("2008-09-01"), color=COLOR_CRISIS, linestyle="--", linewidth=1.5, alpha=0.7)
ax.set_xlabel("Month", fontsize=11)
ax.set_ylabel("Mean Residual", fontsize=11)
ax.set_title("Average Residual over Time", fontsize=12, fontweight="bold")

plt.suptitle("Regression Diagnostics", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()

# Save figure
fig_diag_path = OUTPUT_DIR / "fig_residual_diagnostics.png"
plt.savefig(fig_diag_path, dpi=150, bbox_inches="tight")
print(f"Figure saved: {fig_diag_path}")

plt.show()

print("What to look for:")
print("  1. Histogram: Residuals should be roughly bell-shaped (symmetric around zero).")
print("  2. Residuals vs Fitted: No obvious fan shape or pattern (would indicate heteroskedasticity).")
print("  3. Residuals over Time: No persistent drift (would suggest time-varying omitted variables).")
print()
print("These diagnostics help justify our model assumptions. HC3 standard errors")
print("already protect against heteroskedasticity, but it's good practice to check.")

---

## Section 7: Parallel Trends Plot

The canonical DiD visualization plots the **treated group average** and **control group average** over time on the same axes.

**Before you look at the plot, predict what you SHOULD see:**

- **Pre-crisis (left of the vertical line):** If parallel trends holds, the two lines should move in parallel — similar ups, similar downs. This is what we need for DiD to be credible.
- **Post-crisis (right of the vertical line):** If the treatment had an effect, the treated line should diverge from the control line — it should jump MORE than the control.
- **The gap that opens up** after the crisis line is the DiD estimate visualized.

Now let's see if the real data matches our prediction...

In [ ]:
# =============================================================================
# PARALLEL TRENDS PLOT — Real REIT firm-level data
# =============================================================================

# Re-define crisis date (so this cell runs independently)
crisis_date = pd.Timestamp("2008-09-01")

# Compute average log_vol for high-leverage and low-leverage firms in each month
high_lev_avg = (
    df[df["Treated"] == 1]
    .groupby("month")["log_vol"]
    .mean()
    .reset_index()
)
high_lev_avg.columns = ["month", "avg_log_vol_treated"]

low_lev_avg = (
    df[df["Treated"] == 0]
    .groupby("month")["log_vol"]
    .mean()
    .reset_index()
)
low_lev_avg.columns = ["month", "avg_log_vol_control"]

# Merge for plotting
parallel_df = pd.merge(high_lev_avg, low_lev_avg, on="month")

# Plot
fig, ax = plt.subplots(figsize=(13, 6))

# High-leverage group: blue solid line
ax.plot(
    parallel_df["month"],
    parallel_df["avg_log_vol_treated"],
    color=COLOR_TREATED,
    linewidth=2.5,
    marker="o",
    markersize=5,
    label="High leverage (Treated=1, debt_at > median)",
)

# Low-leverage group: red dashed line
ax.plot(
    parallel_df["month"],
    parallel_df["avg_log_vol_control"],
    color=COLOR_CONTROL,
    linewidth=2.5,
    marker="s",
    markersize=5,
    linestyle="--",
    label="Low leverage (Treated=0, debt_at ≤ median)",
)

# Vertical dashed line at crisis date
ax.axvline(x=crisis_date, color=COLOR_CRISIS, linestyle="--", linewidth=2, label="Sep 2008 crisis")
ax.text(crisis_date + pd.DateOffset(days=30),
        ax.get_ylim()[0] + 0.05,
        "Lehman\nCollapse",
        color=COLOR_CRISIS, fontsize=10, va="bottom", fontweight="bold")

# Shade pre-period
ax.axvspan(parallel_df["month"].min(), crisis_date,
           alpha=0.05, color="green", label="Pre-period (check parallel trends here)")

ax.set_xlabel("Month", fontsize=12)
ax.set_ylabel("Average log(Realized Volatility)", fontsize=12)
ax.set_title("Parallel Trends Plot: High-Leverage vs. Low-Leverage REITs",
             fontsize=14, fontweight="bold")
ax.legend(fontsize=10, loc="upper left")
plt.tight_layout()

# Save figure
fig03_path = OUTPUT_DIR / "fig03_parallel_trends.png"
plt.savefig(fig03_path, dpi=150, bbox_inches="tight")
print(f"Figure saved: {fig03_path}")

plt.show()

### Interpretation Checkpoint: Does Parallel Trends Hold?

Look at the green-shaded pre-period carefully and answer:

- **Do both lines track together before Sep 2008?** The two lines move in roughly the same direction with similar timing. They don't need to be at the *same level* — they just need to move *in parallel* (similar slopes, similar ups and downs).

- **What happens right at the crisis?** Both lines spike sharply — that's the common shock (β₂ ≈ +1.31). But the blue line (high leverage) jumps *higher* than the red line. That extra gap = the DiD effect (β₃ ≈ +0.13).

- **Does the gap persist after the crisis?** Yes — the separation between the lines continues in the post-period, suggesting the leverage amplification wasn't just a one-month blip.

- **Is this Scenario A or Scenario B from Section 2?** This looks like **Scenario A** — the pre-period lines are roughly parallel, supporting DiD credibility. We'll confirm with a formal placebo test next.

---

## Section 8: Placebo Test

A **placebo test** (also called a falsification test) is a powerful way to check whether our DiD result is real.

**The logic:**
> *If we move the crisis date to a time when nothing special happened — say, September 2007 — and we still find a large DiD coefficient, that would suggest our result is spurious (not caused by the actual crisis).*

**What we want to see:**

- **Real DiD** (Sep 2008): Large, statistically significant positive coefficient → the crisis genuinely amplified volatility for high-leverage REITs more than low-leverage ones.
- **Placebo DiD** (Sep 2007): Small coefficient, NOT statistically significant → before the crisis, the two leverage groups were not diverging.

If the placebo test **passes** (near-zero, insignificant), our real DiD result is more credible. If it **fails** (large, significant), we have a problem — the treated and control groups may have been on different pre-trends.

In [ ]:
# =============================================================================
# PLACEBO TEST — Fake crisis date one year before the real crisis
# =============================================================================

# Re-define crisis dates so this cell runs independently
real_crisis_date  = pd.Timestamp("2008-09-01")   # Real event date
fake_crisis_date  = pd.Timestamp("2007-09-01")   # Placebo date — one year earlier

# --- Build placebo dataset using only the PRE-REAL-CRISIS period ---
# We restrict to months before the real crisis to avoid contamination
df_placebo = df[df["month"] < real_crisis_date].copy()

# Treated is already in the data (same pre-determined leverage classification)
# Post indicator now uses the FAKE crisis date
df_placebo["Post_Placebo"] = (df_placebo["month"] >= fake_crisis_date).astype(int)

# Placebo DiD interaction term
df_placebo["DiD_Placebo"] = df_placebo["Treated"] * df_placebo["Post_Placebo"]

# Run the placebo regression on log_vol
placebo_model = smf.ols("log_vol ~ Treated + Post_Placebo + DiD_Placebo",
                         data=df_placebo).fit(cov_type="HC3")

# Extract placebo DiD coefficient
placebo_did_coef = placebo_model.params["DiD_Placebo"]
placebo_did_se   = placebo_model.bse["DiD_Placebo"]
placebo_did_pval = placebo_model.pvalues["DiD_Placebo"]

print("=== Placebo Regression Summary ===")
print(placebo_model.summary())
print()
print(f"Placebo DiD coefficient (fake Sep 2007 crisis): {placebo_did_coef:+.4f}")
print(f"  Standard Error: {placebo_did_se:.4f}")
print(f"  p-value:        {placebo_did_pval:.4f}")
print()
print("Expected: small and statistically insignificant (p > 0.1).")

In [ ]:
# =============================================================================
# COMPARISON TABLE — Real DiD vs Placebo DiD
# =============================================================================

# Build a side-by-side comparison table
comparison_data = {
    "Event Date":    ["Sep 2008 (Real crisis)", "Sep 2007 (Placebo)"],
    "DiD Coefficient (β₃)": [f"{did_coef:+.4f}", f"{placebo_did_coef:+.4f}"],
    "Standard Error":       [f"{did_se:.4f}",         f"{placebo_did_se:.4f}"],
    "p-value":              [f"{did_pval:.4f}",        f"{placebo_did_pval:.4f}"],
    "Significant at 5%?":   [
        "YES ***" if did_pval < 0.05 else "No",
        "YES" if placebo_did_pval < 0.05 else "No",
    ],
}

comparison_table = pd.DataFrame(comparison_data)

print("=== Real DiD vs. Placebo DiD — Side-by-Side ===")
print(comparison_table.to_string(index=False))

print()
print("=== Interpretation ===")
if placebo_did_pval > 0.10:
    print("PLACEBO PASSES")
    print(f"  The placebo DiD ({placebo_did_coef:+.4f}) is small and insignificant (p = {placebo_did_pval:.3f}).")
    print("  Before the real crisis, high- and low-leverage firms were NOT diverging in volatility.")
    print("  This supports the parallel trends assumption and makes the real DiD more credible.")
else:
    print("PLACEBO FAILS")
    print(f"  The placebo DiD ({placebo_did_coef:+.4f}) is significant (p = {placebo_did_pval:.3f}).")
    print("  This suggests the two leverage groups had pre-existing trends, which could bias DiD.")

print()
print(f"Real DiD coefficient:    {did_coef:+.4f}  (p = {did_pval:.4f})  ← crisis amplification")
print(f"Placebo DiD coefficient: {placebo_did_coef:+.4f}  (p = {placebo_did_pval:.4f})  ← no pre-trend divergence")
print()
print("Bottom line: The effect is concentrated at the REAL event date, not the fake one.")

### What Does the Placebo Test Tell Us?

**If the placebo passes (near-zero, p > 0.10):**
Before the real crisis, the two leverage groups were indeed on parallel tracks in terms of volatility. The significant DiD at Sep 2008 is not some pre-existing trend — it genuinely reflects the crisis impact on highly leveraged firms. This is the result we hope to see.

**If the placebo fails (large and significant):**
The treated and control groups were already diverging before the crisis. This means our DiD estimate is likely biased — it's picking up a pre-existing difference, not a causal crisis effect. You would need to reconsider the control group selection or switch to a more flexible method (e.g., event study with leads and lags).

Placebo tests are one of the most important robustness checks in applied econometrics. Reviewers and graders will look for them.

---

## Section 9: Summary

Let's pull all the key results together into one clean table and recap the main takeaways.

In [ ]:
# =============================================================================
# FINAL SUMMARY TABLE — All key DiD results
# =============================================================================

print("=" * 70)
print("WEEK 11 DiD DEMO — KEY RESULTS SUMMARY")
print("=" * 70)
print()
print("Dataset:          reit_firm_panel.csv")
print("Outcome:          log_vol (log of monthly realized return volatility)")
print(f"Treatment group:  High-leverage firms (debt_at > median, Treated=1)")
print(f"Control group:    Low-leverage firms  (debt_at ≤ median, Treated=0)")
print(f"Treatment firms:  ~55 (high leverage)")
print(f"Control firms:    ~56 (low leverage)")
print(f"Crisis date:      September 2008 (Lehman Brothers collapse)")
print(f"N observations:   {int(did_model.nobs)}")
print()
print("-" * 70)
print("MANUAL DiD (Section 5)")
print("-" * 70)
print(f"  High-leverage change (After − Before): {change_treated_firms:+.4f}")
print(f"  Low-leverage  change (After − Before): {change_control_firms:+.4f}")
print(f"  DiD estimate:                          {did_manual:+.4f}")
print()
print("-" * 70)
print("DiD REGRESSION (Section 6) — OLS with HC3 SE")
print("-" * 70)
print(f"  α  Intercept (low-lev baseline log_vol): {intercept_coef:+.4f}")
print(f"  β₁ Treated (pre-existing leverage gap):  {treated_coef:+.4f}")
print(f"  β₂ Post (common crisis volatility spike): {post_coef:+.4f}  ***")
print(f"  β₃ DiD (CAUSAL EFFECT — leverage amplif.): {did_coef:+.4f}  SE={did_se:.4f}  p={did_pval:.4f}")
print(f"     exp(DiD) = {np.exp(did_coef):.3f}  →  ~{(np.exp(did_coef)-1)*100:.1f}% extra volatility amplification")
stat_sig = "*** Statistically significant at 0.1%" if did_pval < 0.001 else "** Statistically significant at 5%" if did_pval < 0.05 else "Not statistically significant"
print(f"     → {stat_sig}")
print(f"  R-squared: {did_model.rsquared:.4f}")
print()
print("-" * 70)
print("PLACEBO TEST (Section 8) — Fake crisis: Sep 2007")
print("-" * 70)
print(f"  β₃ Placebo DiD:  {placebo_did_coef:+.4f}  SE={placebo_did_se:.4f}  p={placebo_did_pval:.4f}")
placebo_result = "PASS — near zero and insignificant" if placebo_did_pval > 0.10 else "FAIL — unexpectedly significant"
print(f"     → Placebo test: {placebo_result}")
print()
print("=" * 70)

### Five Key Takeaways

1. **DiD isolates causal effects from common trends.** The 2008 crisis caused ALL REITs to experience a massive volatility spike (Post = +1.3074). DiD strips this common shock out and isolates the **extra** volatility amplification borne by high-leverage firms specifically.

2. **The DiD coefficient (β₃ = +0.1329, p = 0.0007) is the causal estimate — if parallel trends holds.** exp(0.1329) ≈ 1.14 means high-leverage REITs experienced approximately 14% more volatility amplification than low-leverage peers. The leverage channel — refinancing risk, margin calls, credit downgrades — explains why.

3. **Parallel trends is an assumption, not a fact.** Always plot the pre-period trends and discuss whether the two groups tracked each other before the event. You cannot prove the assumption, but you can provide evidence for or against it.

4. **Placebo tests are essential for credibility.** Our placebo DiD is small and statistically insignificant (p > 0.1) — the two leverage groups were NOT diverging before the crisis. This strongly supports the parallel trends assumption.

5. **DiD is everywhere in applied economics.** Minimum wage laws, policy rollouts, regulatory changes, natural disasters — any time a shock hits some groups but not others at a specific point in time, DiD is a natural tool.

---

### Connection to Assignment 11

In **Assignment 11**, you will apply this exact framework to the same `reit_firm_panel.csv` dataset. You will:

- Use `log_vol` as the outcome variable (log of monthly realized volatility)
- Confirm the treatment assignment (Treated = 1 for high pre-crisis leverage)
- Create the `DiD` interaction variable (`Treated × Post`)
- Compute the manual 2×2 DiD estimate using `log_vol` means
- Run the DiD regression and interpret every coefficient
- Produce the parallel trends plot and assess whether the assumption holds
- Run a placebo test using a fake crisis date of September 2007

The workflow you followed in this demo is exactly the template for your assignment.